In [1]:
import sys
import os 

project_root = "/Users/janikwahrheit/Library/CloudStorage/OneDrive-Persönlich/01_Studium/01_Bachelor/Bachelorarbeit/Code"

sys.path.append(project_root)

<h1> Training Pipeline </h1>

In [2]:
from utils.analytics import eval_fit_methods
import networks
import estimator.stable_estimators as se
from scipy.stats import levy_stable
import matplotlib.pyplot as plt 
import seaborn as sns 
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import pandas as pd
import pickle
import numpy as np
import torch
import torch.nn as nn
import utils.optimreg as optimreg
from data.dataloaders import get_mnist_loaders, get_cifar_loaders

In [3]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

<h2> Configs </h2>

In [6]:
RUNS = 10
# REGULARIZERS = [None, "lasso", "hill", "parabolic_hill", "xiao", "decay"]
REGULARIZERS = ["xiao", "decay"]
LAMBDA_SEARCH = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]

ARCHITECTURES = {
    "fc3_mnist": [784, 256, 256, 256, 10],
    "fc3_cifar": [32*32*3, 256, 256, 256, 10],
    "fc10_mnist": [784] + [256]*10 + [10],
    "fc10_cifar": [32*32*3] + [256]*10 + [10]
}

EPOCHS = {
    "fc3_mnist_sgd": 10,
    "fc3_mnist_adam": 5,
    "fc10_mnist": 200,
    "fc3_cifar": 20,
    "fc10_cifar": 200
}

TUNING_EPOCHS = {
    "fc3_mnist": 2,   
    "fc10_mnist": 5,  
    "fc3_cifar": 5,
    "fc10_cifar": 5
}


<h2> Hyperparameter Tuning </h2> 

In [7]:
def tune_lambda(dataset_name, arch_key, optimizer_name, train_loader, val_loader, reg):
    architecture = ARCHITECTURES[arch_key]

    if reg is None:
        return 0.0  

    best_acc = -1.0
    best_lambda = LAMBDA_SEARCH[0]

    for lam in LAMBDA_SEARCH:
        model = networks.FCNet(
            layer_sizes=architecture,
            activation='relu',
            weight_init='gaussian'
        ).to(device)

        optimizer = optimreg.get_optimizer(model, optimizer_name=optimizer_name, lr=1e-3)
        epochs = TUNING_EPOCHS[arch_key]

        networks.train(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            epochs=epochs,
            model_name=f"tuning_{arch_key}_{reg}_{lam}_{optimizer_name}_{dataset_name}",
            logging=False,
            run=0,  
            regularizer=reg,
            lambda_reg=lam,
            architecture=architecture
        )

        acc = networks.evaluate(model, val_loader)
        if acc > best_acc:
            best_acc = acc
            best_lambda = lam

    print(f"Best lambda for {arch_key} | {reg} | {optimizer_name}: {best_lambda}")
    return best_lambda

<h2> Training Runs </h2>

In [8]:
accuracies_dict = {}

In [ ]:
def run_experiment(dataset_name, arch_key, optimizer_name, train_loader, val_loader, test_loader):
    architecture = ARCHITECTURES[arch_key]

    for reg in REGULARIZERS:
        lam = tune_lambda(dataset_name, arch_key, optimizer_name, train_loader, val_loader, reg)

        accuracies = []

        for run in range(RUNS):
            print("\n==============================================")
            print(f"Dataset: {dataset_name}")
            print(f"Model: {arch_key}")
            print(f"Regularizer: {reg}")
            print(f"Lambda: {lam}")
            print(f"Optimizer: {optimizer_name}")
            print(f"Run: {run}")
            print("==============================================\n")

            model = networks.FCNet(
                layer_sizes=architecture,
                activation='relu',
                weight_init='gaussian'
            ).to(device)

            optimizer = optimreg.get_optimizer(model, optimizer_name=optimizer_name, lr=1e-3)
            epochs = EPOCHS[arch_key+f"_{optimizer_name}"]
            name = f"{arch_key}_{reg}_{optimizer_name}_{dataset_name}"

            networks.train(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                optimizer=optimizer,
                epochs=epochs,
                model_name=name,
                logging=True,
                run=run,
                regularizer=reg,
                lambda_reg=lam,
                architecture=architecture
            )

            acc = networks.evaluate(model, test_loader)
            accuracies.append(acc)

        accuracies_dict[name] = {"accuracies" : accuracies, 
                                 "labmda": lam}
        print("\n================================")
        print(f"Finished: {arch_key} | Reg: {reg} | λ={lam} | Optimizer: {optimizer_name}")
        print("Accuracies:", accuracies)
        print("Mean:", np.mean(accuracies))
        print("Std:", np.std(accuracies))
        print("================================\n")




In [10]:
# OPTIMIZERS = ["sgd", "adam"]
OPTIMIZERS = ["adam"]
BATCH_SIZE = 32


In [11]:
# MNIST
train_loader, val_loader, test_loader = get_mnist_loaders(batch_size=BATCH_SIZE)
for opt in OPTIMIZERS:
    print(f"\n===== Running MNIST FC3 | Optimizer: {opt} =====")
    run_experiment("mnist", "fc3_mnist", opt, train_loader, val_loader, test_loader)

    # print(f"\n===== Running MNIST FC10 | Optimizer: {opt} =====")
    # run_experiment("mnist", "fc10_mnist", opt, train_loader, val_loader, test_loader)



===== Running MNIST FC3 | Optimizer: adam =====
Epoch 1/2 | Train Loss: -4.7069 | Train Acc: 34.97% | Val Loss: 2.2990 | Val Acc: 11.38%
Epoch 2/2 | Train Loss: -10.8114 | Train Acc: 11.34% | Val Loss: 2.2993 | Val Acc: 11.38%
Test Accuracy: 11.38%
Epoch 1/2 | Train Loss: -4.5037 | Train Acc: 36.08% | Val Loss: 2.2986 | Val Acc: 11.38%
Epoch 2/2 | Train Loss: -10.9506 | Train Acc: 11.58% | Val Loss: 2.2997 | Val Acc: 11.38%
Test Accuracy: 11.38%
Epoch 1/2 | Train Loss: -4.5503 | Train Acc: 35.89% | Val Loss: 2.2993 | Val Acc: 11.38%
Epoch 2/2 | Train Loss: -10.8894 | Train Acc: 11.19% | Val Loss: 2.2998 | Val Acc: 11.38%
Test Accuracy: 11.38%
Epoch 1/2 | Train Loss: -4.6680 | Train Acc: 35.18% | Val Loss: 2.2989 | Val Acc: 10.32%
Epoch 2/2 | Train Loss: -10.8653 | Train Acc: 11.62% | Val Loss: 2.2994 | Val Acc: 11.38%
Test Accuracy: 11.38%
Epoch 1/2 | Train Loss: -4.6880 | Train Acc: 35.48% | Val Loss: 2.2989 | Val Acc: 11.38%
Epoch 2/2 | Train Loss: -11.0775 | Train Acc: 11.38% | Val

In [ ]:

# CIFAR
train_loader, val_loader, test_loader = get_cifar_loaders(batch_size=BATCH_SIZE)
for opt in OPTIMIZERS:
    print(f"\n===== Running CIFAR FC3 | Optimizer: {opt} =====")
    run_experiment("cifar", "fc3_cifar", opt, train_loader, val_loader, test_loader)

    print(f"\n===== Running CIFAR FC10 | Optimizer: {opt} =====")
    run_experiment("cifar", "fc10_cifar", opt, train_loader, val_loader, test_loader)